# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) available via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview
Review available record sets and their `@id`s and inspect available fields. All references use Croissant `@id` identifiers.

In [ ]:
# List available record sets by their @id and fields
record_sets = list(dataset.record_sets)

print(f"Number of record sets found: {len(record_sets)}")
for recset in record_sets:
    print(f"RecordSet @id: {recset['@id']}")
    print(f"  Name: {recset.get('name','')}")
    print(f"  Description: {recset.get('description','')}")
    fields = recset.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields and their @id:")
    for f in fields:
        if isinstance(f, dict) and '@id' in f:
            print(f"    - {f['@id']}")
        else:
            print(f"    - {f}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Reference all record sets and fields by their `@id`.

In [ ]:
# Define the record set(s) you want to extract (use their @id!)
# List of RecordSet @ids for this dataset (found in the previous overview cell).
# For this dataset the main record set(s) are:
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0-records'  # You may need to replace with correct one as found above

record_sets_to_extract = [main_record_set_id]
dataframes = {}

for record_set_id in record_sets_to_extract:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by value or normalizing numeric fields. This example demonstrates how to filter, normalize, and group data referencing field `@id`s (column names as imported by Croissant).

In [ ]:
# Select the DataFrame to analyze
record_set_id = main_record_set_id

# Choose a numeric field by @id or column name
# (Example: 'cr:numberOfPeptides' or similar, replace below with your actual column name.)
numeric_field = None
for col in dataframes[record_set_id].columns:
    if 'peptide' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = dataframes[record_set_id].columns[0]  # fallback
print(f"Numeric field used for analysis: {numeric_field}")

# Cast column to float if possible
df = dataframes[record_set_id].copy()
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].mean()  # or choose a value like 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field for filtered records
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (e.g., 'cr:accession' or 'cr:description' or similar)
candidates = [col for col in df.columns if 'desc' in col.lower() or 'accession' in col.lower()]
if len(candidates) > 0:
    group_field = candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset for deeper insight.

Let's plot the distribution of the selected numeric field and, if possible, grouped means by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 mass spectrometry dataset using `mlcroissant`, accessed fields and record sets by their `@id`, performed filtering and normalization, and visualized key variables. For deeper analysis, consult the Croissant metadata for specific field meanings and structure, always referencing by `@id`.